# Sample Notebook: OLMoEarth Embedding Generation

This notebook shows how to use the Pixelverse model registry to load OLMoEarth and generate embeddings from Sentinel-2 time-series chips.

Current wrapper contract:
- image input: `B, T, C, H, W`
- timestamps input: `B, T, 3` as `[day, month, year]`
- output: pooled embedding `B, D` (global embedding per chip)


In [ ]:
import numpy as np
import torch
import xarray as xr

import pixelverse as pv

In [ ]:
pv.list_models()

## Pick a variant

Available OLMoEarth variants in the registry:
- `olmoearth_nano`
- `olmoearth_tiny`
- `olmoearth_base`
- `olmoearth_large`


In [ ]:
MODEL_NAME = "olmoearth_nano"
model, transforms = pv.create_model(MODEL_NAME)  # downloads weights on first run
weights = pv.get_weights(MODEL_NAME)
model.eval()
weights.meta

## Sentinel-2 input requirements

OLMoEarth expects 12 Sentinel-2 bands in this order (channel dimension `C`):

`B02, B03, B04, B08, B05, B06, B07, B8A, B11, B12, B01, B09`

The helper below converts an `xarray.Dataset` with variables named exactly like those bands and dimensions `(time, y, x)` into Pixelverse OLMoEarth inputs.


In [ ]:
OLMOEARTH_BANDS = [
    "B02",
    "B03",
    "B04",
    "B08",
    "B05",
    "B06",
    "B07",
    "B8A",
    "B11",
    "B12",
    "B01",
    "B09",
]


def build_olmoearth_inputs_from_xarray(ds: xr.Dataset) -> tuple[torch.Tensor, torch.Tensor]:
    missing = [band for band in OLMOEARTH_BANDS if band not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required bands: {missing}")
    if "time" not in ds.dims:
        raise ValueError("Dataset must have a time dimension")

    # (band, time, y, x) -> (time, band, y, x) -> add batch dim -> (B, T, C, H, W)
    arr = ds[OLMOEARTH_BANDS].to_array(dim="band").transpose("time", "band", "y", "x").values
    x = torch.from_numpy(arr).unsqueeze(0).float()

    t = ds.time.dt
    timestamps_np = np.stack(
        [
            t.day.values.astype(np.int64),
            (t.month.values - 1).astype(np.int64),  # OLMoEarth uses zero-indexed months
            t.year.values.astype(np.int64),
        ],
        axis=-1,
    )
    timestamps = torch.from_numpy(timestamps_np).unsqueeze(0)
    return x, timestamps

## Minimal synthetic example (runnable everywhere)

This verifies the registry call + normalization + forward pass without needing to fetch STAC data.


In [ ]:
B, T, C, H, W = 1, 4, 12, 16, 16
# Synthetic S2-like reflectance values (uint16 range), cast to float for transforms
x = torch.randint(0, 10000, (B, T, C, H, W), dtype=torch.int32).float()
timestamps = torch.tensor(
    [[[15, 0, 2024], [15, 1, 2024], [15, 2, 2024], [15, 3, 2024]]], dtype=torch.long
)

with torch.no_grad():
    embeddings = model(transforms(x), timestamps=timestamps)

embeddings.shape

## Example with an xarray Dataset

If you already have an `xarray.Dataset` with the 12 required S2 bands and a `time` coordinate, convert and run inference like this.


In [ ]:
# Example (not executed here):
# ds_s2 = ...  # xr.Dataset with variables B02..B12/B8A/B01/B09 and dims (time, y, x)
# x, timestamps = build_olmoearth_inputs_from_xarray(ds_s2)
# with torch.no_grad():
#     embeddings = model(transforms(x), timestamps=timestamps)
#
# embeddings is shape (B, D); for one chip batch that is (1, embed_dim)


## Save embeddings

Since the current OLMoEarth wrapper returns a global embedding per chip (`B, D`), a simple output format is NumPy or a tabular/xarray record keyed by chip id.


In [ ]:
np.save("olmoearth_chip_embeddings.npy", embeddings.cpu().numpy())